In [1]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


@tool
def get_weather(city: str):
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

ValueError: Function must have a docstring if description not provided.

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint


@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

In [8]:
@tool
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华 氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报\n\nArgs:\n    city: 城市\n    units: 
气温单位，可选：celsius-摄氏度，fahrenheit-华氏度\n    include_forecast: 是否包含未来五日的天气预报',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'units': {'default': 'celsius', 'type': 'string'},
                'include_forecast': {'default': False, 'type': 'boolean'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

In [11]:
@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if units == "celsius" else "华 氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool("getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'getWeather', 'description': '天气查询工具', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}


## 自定义args_schema

### 方式1：使用Pydantic模型定义

In [27]:
from typing import Literal
from pydantic import Field, BaseModel


class WeatherInput(BaseModel):
    city: str = Field(
        description="具体城市",
        default="Peking"
    )
    unit: Literal["celsius", "fahrenheit", "kelvin"] = Field(
        description="气温单位"
    )


rprint(WeatherInput(city="Shanghai", unit="kelvin"))


@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

WeatherInput(city='Shanghai', unit='kelvin')

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'default': 'Peking', 'description': '具体城市', 'type': 'string'},
                'unit': {
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit', 'kelvin'],
                    'type': 'string'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

## 方式2：使用Json Schema定义

In [29]:
json_schema = {
    'properties': {
        'city': {'default': 'Peking', 'description': '具体城市', 'type': 'string'},
        'unit': {
            'description': '气温单位',
            'enum': ['celsius', 'fahrenheit', 'kelvin'],
            'type': 'string'
        }
    },
    'required': ['unit'],
    'type': 'object'
}
@tool(args_schema=json_schema)
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'default': 'Peking', 'description': '具体城市', 'type': 'string'},
                'unit': {
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit', 'kelvin'],
                    'type': 'string'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}